# LLM judges for prediction-flipping validation candidates

Prepare `judge_inputs.csv` locally using `prepare_inputs("artifacts/attacks")`, upload that one CSV as a Kaggle Dataset and attach it here. It includes all distinct paths to prediction-flipping variants, using five columns: `source_id`, `original`, `original_label`, `candidate`, `path`. Each path is one line with numbered C/W/S steps separated by `->`. No classifier checkpoint is needed.

Use a T4 x2 session with Internet enabled. Qwen and SEA-LION judge independently on separate GPUs. Gemma judges disagreements, uncertain answers and malformed responses after the first two models exit. All outputs remain suggestions for human verification; neither accepted nor rejected paths are removed.

Completed rows are reused on rerun, including unresolved outputs. Restore the judge CSVs into `OUTPUT_DIR` to resume in a new Kaggle session. Use another output directory when changing the rubric or generation settings.

All four output CSVs contain the five input columns plus `candidate_label` and `valid`. Results are joined by `source_id` + `path`, without generated IDs or dependence on row order. `valid` combines meaning, edit-level compliance and label preservation as YES/NO/UNSURE. Failed responses remain UNSURE for further review.


In [1]:
GITHUB_REPO_URL = "https://github.com/24520152-ben/LLM-Generated-Multi-Level-Adversarial-Attacks-on-Vietnamese-Hate-Speech-Detection"
# Upload artifacts/judges/judge_inputs.csv as a Kaggle Dataset and attach it.
# Leave blank to find the unique judge_inputs.csv under /kaggle/input.
INPUT_CSV = "/kaggle/input/datasets/hophamsailam/vihsd-attack-pilot-judge-inputs/judge_inputs.csv"
OUTPUT_DIR = "/kaggle/working/artifacts/judges"
JUDGE_DEVICES = {"qwen": "cuda:0", "sealion": "cuda:1", "gemma": "cuda:0"}
JUDGE_OPTIONS = {"max_input_tokens": 8192, "max_new_tokens": 128}

In [2]:
import json
import os
import subprocess
from pathlib import Path

kaggle_working = Path("/kaggle/working")
project_root = Path.cwd()
if kaggle_working.is_dir():
    project_root = kaggle_working / "ie403"
    if not project_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--", GITHUB_REPO_URL, str(project_root)],
            check=True,
        )
elif not (project_root / "src" / "vihsd_attack").is_dir():
    project_root = project_root.parent

source_dir = project_root / "src"
if not (source_dir / "vihsd_attack").is_dir():
    raise RuntimeError(f"Cannot find src/vihsd_attack in {project_root}.")
env = {
    **os.environ,
    "UV_PROJECT_ENVIRONMENT": str(project_root / ".venv"),
    "PYTHONPATH": str(source_dir),
}
if kaggle_working.is_dir() and not env.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:  # noqa: BLE001 - Kaggle raises backend-specific secret errors.
        print("HF_TOKEN secret unavailable; using public model access.")

subprocess.run(
    ["uv", "sync", "--locked", "--python", "3.12"],
    cwd=project_root,
    env=env,
    check=True,
)

from shutil import copyfile

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
if INPUT_CSV:
    input_csv = Path(INPUT_CSV)
else:
    candidates = (
        list(Path("/kaggle/input").rglob("judge_inputs.csv"))
        if kaggle_working.is_dir()
        else [project_root / "artifacts/judges/judge_inputs.csv"]
    )
    if len(candidates) != 1:
        raise RuntimeError("Set INPUT_CSV to the attached judge_inputs.csv path.")
    input_csv = candidates[0]
input_copy = output_dir / "judge_inputs.csv"
if input_csv.resolve() != input_copy.resolve():
    copyfile(input_csv, input_copy)

judge_code = """
import json
import sys
from vihsd_attack.judges.config import JudgeConfig
from vihsd_attack.judges.service import run_judge

run_judge(JudgeConfig(**json.loads(sys.argv[1])))
"""


def judge_command(name):
    """Build an isolated command for one judge.

    Args:
        name: Judge alias from JUDGE_DEVICES.

    Returns:
        Command arguments for the selected model and GPU.
    """
    options = {
        **JUDGE_OPTIONS,
        "input_file": str(input_copy),
        "output_dir": str(output_dir),
        "judge": name,
        "device": JUDGE_DEVICES[name],
    }
    return [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-u",
        "-c",
        judge_code,
        json.dumps(options),
    ]

Cloning into '/kaggle/working/ie403'...
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
Resolved 86 packages in 34ms
 Downloaded pygments
 Downloaded nvidia-cufile
 Downloaded underthesea-core
 Downloaded sentencepiece
 Downloaded networkx
 Downloaded pydantic-core
 Downloaded nvidia-cuda-runtime
 Downloaded rapidfuzz
 Downloaded tokenizers
 Downloaded nvidia-cuda-cccl
 Downloaded hf-xet
 Downloaded sympy
 Downloaded underthesea
 Downloaded cuda-bindings
 Downloaded scikit-learn
 Downloaded nvidia-cuda-cupti
 Downloaded pandas
 Downloaded transformers
 Downloaded numpy
 Downloaded scipy
 Downloaded bitsandbytes
 Downloaded nvidia-cuda-nvrtc
 Downloaded nvidia-nvjitlink
 Downloaded pyarrow
 Downloaded nvidia-nvshmem-cu13
 Downloaded nvidia-cusparse
 Downloaded nvidia-cusparselt-cu13
 Downloaded nvidia-nccl-cu13
 Downloaded nvidia-cufft
 Downloaded nvidia-cusolver
 Downloaded triton
 Downloaded nvidia-cublas
 Downloaded nvidia-cudnn-cu13
 D

In [3]:
import time

# Two independent processes; stop both if either process fails.
jobs = []
try:
    for name in ("qwen", "sealion"):
        jobs.append(subprocess.Popen(judge_command(name), cwd=project_root, env=env))
    while True:
        exit_codes = [job.poll() for job in jobs]
        if any(code not in (None, 0) for code in exit_codes):
            raise RuntimeError(
                f"First-stage judge failed: {exit_codes}; rerun to resume."
            )
        if all(code is not None for code in exit_codes):
            break
        time.sleep(1)
finally:
    for job in jobs:
        if job.poll() is None:
            job.terminate()
            job.wait()

Judge sealion:   0%|          | 0/1773 [00:00<?, ?it/s]

Judge: aisingapore/Llama-SEA-LION-v3.5-8B-R, 8-bit on cuda:1


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 427/427 [02:15<00:00,  3.16it/s]


Judge: Qwen/Qwen3.5-9B, 8-bit on cuda:0


Judge sealion:   0%|          | 2/1773 [00:08<2:06:25,  4.28s/it][transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
Judge sealion: 100%|██████████| 1773/1773 [1:42:10<00:00,  3.46s/it]


Saved judge: /kaggle/working/artifacts/judges/sealion.csv


Judge qwen: 100%|██████████| 1773/1773 [2:49:35<00:00,  5.74s/it]


Saved judge: /kaggle/working/artifacts/judges/qwen.csv


In [4]:
# The first two processes have exited, freeing GPU memory for Gemma.
subprocess.run(judge_command("gemma"), cwd=project_root, env=env, check=True)
merge_code = """
import sys
from vihsd_attack.judges.service import merge_judgments

merge_judgments(sys.argv[1], sys.argv[2])
"""
subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-u",
        "-c",
        merge_code,
        str(input_copy),
        str(output_dir),
    ],
    cwd=project_root,
    env=env,
    check=True,
)

Loading weights: 100%|██████████| 677/677 [01:27<00:00,  7.71it/s]


Judge: google/gemma-4-12B-it, 4-bit on cuda:0


Judge gemma: 100%|██████████| 925/925 [1:19:47<00:00,  5.18s/it]


Saved judge: /kaggle/working/artifacts/judges/gemma.csv
LLM suggestions only (all still need human review): {'NO': 1601, 'YES': 172}


CompletedProcess(args=['uv', 'run', '--no-sync', 'python', '-u', '-c', '\nimport sys\nfrom vihsd_attack.judges.service import merge_judgments\n\nmerge_judgments(sys.argv[1], sys.argv[2])\n', '/kaggle/working/artifacts/judges/judge_inputs.csv', '/kaggle/working/artifacts/judges'], returncode=0)